In [1]:
import threading
import requests

In [2]:
connection_semaphore = threading.Semaphore(3)

In [4]:
def fetch_currency_conversion(data):
    amount, from_currency, to_currency = data
    url = f"https://api.frankfurter.app/latest?amount={amount}&from={from_currency}&to={to_currency}"
    connection_semaphore.acquire()
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            rate = data['rates'][to_currency]
            return f"{amount} {from_currency} = {rate} {to_currency}"
        else:
            return "Failed to fetch data from API"
    finally:
        connection_semaphore.release()

In [5]:
def thread_function(data):
    result = fetch_currency_conversion(data)
    print(result)

In [6]:
conversion_requests = [
    (100, "USD", "EUR"),
    (200, "EUR", "GBP"),
    (300, "GBP", "USD"),
    (150, "USD", "JPY"),
    (250, "JPY", "INR"),
    (350, "INR", "CNY"),
    (400, "CNY", "AUD"),
    (500, "AUD", "CAD"),
    (600, "CAD", "CHF"),
    (700, "CHF", "USD")
]


In [8]:
threads = []
for request in conversion_requests:
    thread = threading.Thread(target=thread_function, args=(request,))
    threads.append(thread)
    thread.start()
for thread in threads:
    thread.join()

print("All currency conversions have been fetched.")

100 USD = 85.73 EUR
300 GBP = 403.21 USD
200 EUR = 173.58 GBP
250 JPY = 145.09 INR
350 INR = 29.239 CNY
150 USD = 22191 JPY
500 AUD = 449.06 CAD
600 CAD = 349.3 CHF
400 CNY = 85.02 AUD
700 CHF = 878.11 USD
All currency conversions have been fetched.
